# Layer 4 — Geospatial Tile Serving

Prepares all map layers for the web application (Layer 5):
1. **GEE tile endpoints** — raster LULC, predictions, suitability via `getMapId()`
2. **PMTiles** — OSM vectors packaged for serverless serving
3. **Tile manifest** — JSON config consumed by MapLibre GL in Layer 5
4. **Firebase Hosting** — static deploy target for tiles + app

GEE tiles are free and instant (no download needed). PMTiles serve from any static host.

## 0. Setup

In [ ]:
import ee
import json
from pathlib import Path

ee.Authenticate()
ee.Initialize(project='vivid-brand-182819')
print('Earth Engine initialized:', ee.String('connected').getInfo())

BBOX = {'west': 91.55, 'south': 26.10, 'east': 91.88, 'north': 26.25}
aoi = ee.Geometry.Rectangle([BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']])

LULC_PALETTE = ['e41a1c', '4daf4a', '377eb8', 'c2a06a']  # Urban, Veg, Water, Barren
LULC_LABELS = ['Urban', 'Vegetation', 'Water', 'Barren']
SCALE = 30
DRIVE_FOLDER = 'guwahati_urban_prediction'
CENTER = [26.1445, 91.7362]  # lat, lon

print('Setup complete')

## 1. Rebuild LULC maps & predictions in GEE

In [ ]:
# ── Composite + classification pipeline (same as Layers 2-3) ──

def cloud_mask_landsat(image):
    qa = image.select('QA_PIXEL')
    return image.updateMask(qa.bitwiseAnd(1 << 3).eq(0).And(qa.bitwiseAnd(1 << 4).eq(0)))

def add_indices(image, nir, red, swir, green):
    ndvi = image.normalizedDifference([nir, red]).rename('NDVI')
    ndbi = image.normalizedDifference([swir, nir]).rename('NDBI')
    ndwi = image.normalizedDifference([green, nir]).rename('NDWI')
    return image.addBands([ndvi, ndbi, ndwi])

def scale_l8(image):
    return image.addBands(
        image.select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']).multiply(0.0000275).add(-0.2),
        overwrite=True)

def scale_l5(image):
    return image.addBands(
        image.select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']).multiply(0.0000275).add(-0.2),
        overwrite=True)

def build_composite(start, end, sensor):
    if sensor == 'L8':
        col_id, sfn = 'LANDSAT/LC08/C02/T1_L2', scale_l8
        bands = ['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7']
        nir, red, swir, green = 'SR_B5', 'SR_B4', 'SR_B6', 'SR_B3'
    else:
        col_id, sfn = 'LANDSAT/LT05/C02/T1_L2', scale_l5
        bands = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7']
        nir, red, swir, green = 'SR_B4', 'SR_B3', 'SR_B5', 'SR_B2'
    _n, _r, _s, _g = nir, red, swir, green
    def _add(img, _n=_n, _r=_r, _s=_s, _g=_g):
        return add_indices(img, _n, _r, _s, _g)
    col = (ee.ImageCollection(col_id).filterBounds(aoi).filterDate(start, end)
           .filter(ee.Filter.lt('CLOUD_COVER', 30)).map(cloud_mask_landsat).map(sfn).map(_add))
    return col.select(bands + ['NDVI','NDBI','NDWI']).median().clip(aoi).toFloat()

CLASSIFY_BANDS = ['NDVI', 'NDBI', 'NDWI']

def generate_training(composite):
    ndvi, ndbi, ndwi = composite.select('NDVI'), composite.select('NDBI'), composite.select('NDWI')
    labels = (ee.Image(0)
        .where(ndbi.gt(0.0).And(ndvi.lt(0.25)), 0)
        .where(ndvi.gt(0.35).And(ndbi.lt(0.0)), 1)
        .where(ndwi.gt(0.1).And(ndvi.lt(0.15)), 2)
        .where(ndvi.lt(0.15).And(ndbi.lt(0.05)).And(ndwi.lt(0.0)), 3)
        .rename('label').toUint8())
    mask = ndbi.gt(0.0).And(ndvi.lt(0.25)).Or(ndvi.gt(0.35).And(ndbi.lt(0.0))).Or(
        ndwi.gt(0.1).And(ndvi.lt(0.15))).Or(ndvi.lt(0.15).And(ndbi.lt(0.05)).And(ndwi.lt(0.0)))
    labels = labels.updateMask(mask)
    return composite.addBands(labels).stratifiedSample(
        numPoints=500, classBand='label', region=aoi, scale=SCALE, seed=42, geometries=True)

def proximity_stack(lulc_img):
    dist_w = lulc_img.eq(2).fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_water')
    edge = lulc_img.eq(0).And(lulc_img.eq(0).focal_min(1).Not())
    dist_e = edge.fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_urban_edge')
    viirs = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG').filterBounds(aoi).filterDate('2021-01-01','2021-12-31').select('avg_rad').median().clip(aoi)
    lit = viirs.gt(5).unmask(0)
    dist_r = lit.fastDistanceTransform(256).sqrt().multiply(SCALE).rename('dist_road')
    srtm = ee.Image('USGS/SRTMGL1_003').clip(aoi)
    return dist_w.addBands(dist_e).addBands(dist_r).addBands(
        ee.Terrain.slope(srtm).rename('slope')).addBands(srtm.select('elevation')).toFloat()

# Build epochs
EPOCHS = {
    '2000': ('2000-11-01','2001-04-30','L5'),
    '2005': ('2005-11-01','2006-04-30','L5'),
    '2010': ('2010-11-01','2011-04-30','L5'),
    '2016': ('2016-11-01','2017-04-30','L8'),
    '2021': ('2021-11-01','2022-04-30','L8'),
}

composites = {}
for lbl, (s, e, sen) in EPOCHS.items():
    composites[lbl] = build_composite(s, e, sen)
    print(f'[Composite] {lbl}')

# Classify
samples = generate_training(composites['2021'])
cart = ee.Classifier.smileCart(maxNodes=50).train(samples, 'label', CLASSIFY_BANDS)
lulc_maps = {lbl: comp.select(CLASSIFY_BANDS).classify(cart).toUint8().rename('lulc')
              for lbl, comp in composites.items()}
print(f'Classified {len(lulc_maps)} epochs')

# ── Forecast (same as Layer 3) ──
import numpy as np

def compute_tm(lf, lt):
    tr = lf.multiply(10).add(lt).rename('t')
    pa = ee.Image.pixelArea()
    m = np.zeros((4,4))
    for i in range(4):
        for j in range(4):
            m[i][j] = ee.Number(tr.eq(i*10+j).multiply(pa).reduceRegion(
                ee.Reducer.sum(), aoi, SCALE, maxPixels=1e9).get('t')).getInfo()
    rs = m.sum(axis=1, keepdims=True); rs[rs==0]=1
    return m / rs

print('Computing Markov matrices...')
tms = {}
for e1, e2 in [('2000','2005'),('2005','2010'),('2010','2016'),('2016','2021')]:
    tms[f'{e1}_{e2}'] = compute_tm(lulc_maps[e1], lulc_maps[e2])
    print(f'  {e1}→{e2} done')
avg_tm = np.mean(list(tms.values()), axis=0)

# Train GBT
FEAT_BANDS = ['NDVI','NDBI','NDWI','dist_water','dist_urban_edge','dist_road','slope','elevation']

def change_samples(lf, la, comp, lulc_img):
    prox = proximity_stack(lulc_img)
    feat = comp.select(['NDVI','NDBI','NDWI']).addBands(prox)
    bu = lf.neq(0).And(la.eq(0)).rename('became_urban').toUint8()
    sn = lf.neq(0).And(la.neq(0)).rename('became_urban').multiply(0).toUint8()
    pos = feat.addBands(bu.updateMask(bu)).sample(aoi, SCALE, numPixels=1000, seed=42, geometries=True)
    neg = feat.addBands(sn.updateMask(lf.neq(0).And(la.neq(0)))).sample(aoi, SCALE, numPixels=2000, seed=43, geometries=True)
    return pos.merge(neg)

print('Training GBT...')
s1 = change_samples(lulc_maps['2010'], lulc_maps['2016'], composites['2010'], lulc_maps['2010'])
s2 = change_samples(lulc_maps['2016'], lulc_maps['2021'], composites['2016'], lulc_maps['2016'])
gbt = ee.Classifier.smileGradientTreeBoost(100, shrinkage=0.1, maxNodes=20, seed=42).train(
    s1.merge(s2), 'became_urban', FEAT_BANDS)
gbt_prob = gbt.setOutputMode('PROBABILITY')
print('GBT trained')

# Forecast loop
def forecast_step(cur, comp, tm, gbt_p):
    prox = proximity_stack(cur)
    feat = comp.select(['NDVI','NDBI','NDWI']).addBands(prox)
    suit = feat.classify(gbt_p).rename('urban_prob').updateMask(cur.neq(0))
    pa = ee.Image.pixelArea()
    ca = {c: ee.Number(cur.eq(c).multiply(pa).reduceRegion(
        ee.Reducer.sum(), aoi, SCALE, maxPixels=1e9).get('lulc')).getInfo() for c in range(4)}
    nu = sum(ca[c]*tm[c][0] for c in range(1,4))
    nt = sum(ca[c] for c in range(1,4))
    frac = nu/nt if nt > 0 else 0
    pctl = max(0, min(99, (1-frac)*100))
    thr = ee.Number(suit.reduceRegion(ee.Reducer.percentile([pctl]), aoi, SCALE, maxPixels=1e9).get('urban_prob')).getInfo()
    mask = suit.gt(thr).unmask(0).And(cur.neq(2))
    return cur.where(cur.neq(0).And(mask), 0), nu

predicted = {}
cur = lulc_maps['2021']
for yr in ['2026','2031','2036']:
    print(f'Forecasting {yr}...')
    cur, nu = forecast_step(cur, composites['2021'], avg_tm, gbt_prob)
    predicted[yr] = cur
    a = ee.Number(cur.eq(0).multiply(ee.Image.pixelArea()).reduceRegion(
        ee.Reducer.sum(), aoi, SCALE, maxPixels=1e9).get('lulc')).divide(1e6).getInfo()
    print(f'  {yr} urban: {a:.2f} km²')

# Suitability surface
prox_2021 = proximity_stack(lulc_maps['2021'])
feat_2021 = composites['2021'].select(['NDVI','NDBI','NDWI']).addBands(prox_2021)
suitability = feat_2021.classify(gbt_prob).rename('urban_prob').toFloat()

print('\nAll layers ready for tile serving')

## 2. Generate GEE tile URLs for all raster layers

In [ ]:
# Generate GEE tile URLs for LULC maps (historical + predicted)
# getMapId() returns a tile URL template that MapLibre GL can consume directly

tile_urls = {}

# LULC vis params — categorical map with class colors
lulc_vis = {'min': 0, 'max': 3, 'palette': LULC_PALETTE}

# Historical LULC
for year, lulc in lulc_maps.items():
    map_id = lulc.getMapId(lulc_vis)
    tile_urls[f'lulc_{year}'] = map_id['tile_fetcher'].url_format
    print(f'[Tile] lulc_{year}: ready')

# Predicted LULC
for year, lulc in predicted.items():
    map_id = lulc.getMapId(lulc_vis)
    tile_urls[f'lulc_{year}'] = map_id['tile_fetcher'].url_format
    print(f'[Tile] lulc_{year} (predicted): ready')

# Change detection — urban gain highlighted in red
for e1, e2 in [('2000','2005'),('2005','2010'),('2010','2016'),('2016','2021')]:
    urban_gain = lulc_maps[e1].neq(0).And(lulc_maps[e2].eq(0)).selfMask()
    map_id = urban_gain.getMapId({'palette': ['ff0000']})
    tile_urls[f'urban_gain_{e1}_{e2}'] = map_id['tile_fetcher'].url_format
    print(f'[Tile] urban_gain_{e1}_{e2}: ready')

# Predicted urban gain
for e1, e2 in [('2021','2026'),('2026','2031'),('2031','2036')]:
    prev = lulc_maps.get(e1, predicted.get(e1))
    curr = predicted[e2]
    urban_gain = prev.neq(0).And(curr.eq(0)).selfMask()
    map_id = urban_gain.getMapId({'palette': ['ff6600']})
    tile_urls[f'urban_gain_{e1}_{e2}'] = map_id['tile_fetcher'].url_format
    print(f'[Tile] urban_gain_{e1}_{e2} (predicted): ready')

print(f'\n{len(tile_urls)} tile endpoints generated')

## 3. Generate VIIRS & suitability tile URLs

In [ ]:
# VIIRS night-lights tile URLs
viirs_vis = {'min': 0, 'max': 60, 'palette': ['000000', '0000ff', '00ffff', 'ffff00', 'ff0000']}

for year in [2016, 2019, 2021, 2023]:
    viirs = (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
             .filterBounds(aoi).filterDate(f'{year}-01-01', f'{year}-12-31')
             .select('avg_rad').median().clip(aoi))
    map_id = viirs.getMapId(viirs_vis)
    tile_urls[f'viirs_{year}'] = map_id['tile_fetcher'].url_format
    print(f'[Tile] viirs_{year}: ready')

# Suitability surface — heat map
suit_vis = {'min': 0, 'max': 1, 'palette': ['1a9850','91cf60','d9ef8b','fee08b','fc8d59','d73027']}
map_id = suitability.getMapId(suit_vis)
tile_urls['suitability_2021'] = map_id['tile_fetcher'].url_format
print('[Tile] suitability_2021: ready')

# NDVI (vegetation health) for 2021
ndvi_vis = {'min': -0.1, 'max': 0.8, 'palette': ['d73027','fc8d59','fee08b','d9ef8b','91cf60','1a9850']}
map_id = composites['2021'].select('NDVI').getMapId(ndvi_vis)
tile_urls['ndvi_2021'] = map_id['tile_fetcher'].url_format
print('[Tile] ndvi_2021: ready')

# Elevation (DEM)
dem = ee.Image('USGS/SRTMGL1_003').clip(aoi)
dem_vis = {'min': 40, 'max': 300, 'palette': ['006633','E5FFCC','662A00','D8D8D8','F5F5F5']}
map_id = dem.getMapId(dem_vis)
tile_urls['elevation'] = map_id['tile_fetcher'].url_format
print('[Tile] elevation: ready')

# Slope
slope = ee.Terrain.slope(dem)
slope_vis = {'min': 0, 'max': 30, 'palette': ['00ff00','ffff00','ff0000']}
map_id = slope.getMapId(slope_vis)
tile_urls['slope'] = map_id['tile_fetcher'].url_format
print('[Tile] slope: ready')

print(f'\nTotal tile endpoints: {len(tile_urls)}')

## 4. OSM vectors → PMTiles

In [ ]:
# Convert OSM GeoJSON files to PMTiles for serverless vector tile serving
# tippecanoe runs on Linux (Colab) — install and convert

!apt-get install -y -qq libsqlite3-dev zlib1g-dev 2>/dev/null
!pip install -q pmtiles

# Check if tippecanoe is available, install if not
import shutil
if not shutil.which('tippecanoe'):
    print('Installing tippecanoe...')
    !git clone -q https://github.com/felt/tippecanoe.git /tmp/tippecanoe
    !cd /tmp/tippecanoe && make -j$(nproc) -s 2>/dev/null && cp tippecanoe /usr/local/bin/
    print('tippecanoe installed')
else:
    print('tippecanoe already available')

# Re-download OSM data via osmnx (files from Layer 1 are in Colab runtime, may be gone)
!pip install -q osmnx geopandas

import osmnx as ox
import geopandas as gpd
import os

bbox = (91.55, 26.10, 91.88, 26.25)
osm_files = {}

# Roads
print('\n[OSM] Fetching roads...')
try:
    G = ox.graph_from_bbox(bbox=bbox, network_type='drive')
    edges = ox.graph_to_gdfs(G, nodes=False)
    edges.to_file('osm_roads.geojson', driver='GeoJSON')
    osm_files['roads'] = 'osm_roads.geojson'
    print(f'  {len(edges)} road segments')
except Exception as e:
    print(f'  Failed: {e}')

# Railways
print('[OSM] Fetching railways...')
try:
    rw = ox.features_from_bbox(bbox=bbox, tags={'railway': 'rail'})
    rw.to_file('osm_railways.geojson', driver='GeoJSON')
    osm_files['railways'] = 'osm_railways.geojson'
    print(f'  {len(rw)} features')
except Exception as e:
    print(f'  Failed: {e}')

# Infrastructure POIs
print('[OSM] Fetching infrastructure...')
try:
    infra = ox.features_from_bbox(bbox=bbox, tags={
        'amenity': ['hospital','school','college','university','marketplace'],
        'office': 'government'})
    infra.to_file('osm_infrastructure.geojson', driver='GeoJSON')
    osm_files['infrastructure'] = 'osm_infrastructure.geojson'
    print(f'  {len(infra)} features')
except Exception as e:
    print(f'  Failed: {e}')

# Water
print('[OSM] Fetching water...')
try:
    water = ox.features_from_bbox(bbox=bbox, tags={
        'waterway': ['river','stream','canal'], 'natural': 'water'})
    water.to_file('osm_water.geojson', driver='GeoJSON')
    osm_files['water'] = 'osm_water.geojson'
    print(f'  {len(water)} features')
except Exception as e:
    print(f'  Failed: {e}')

# Convert all GeoJSON to a single PMTiles file
if osm_files:
    input_args = ' '.join([f'-L {name}:{path}' for name, path in osm_files.items()])
    cmd = f'tippecanoe -o osm_guwahati.pmtiles -zg --drop-densest-as-needed --extend-zooms-if-still-dropping {input_args} --force'
    print(f'\nRunning tippecanoe...')
    os.system(cmd)

    if os.path.exists('osm_guwahati.pmtiles'):
        size_mb = os.path.getsize('osm_guwahati.pmtiles') / 1e6
        print(f'[PMTiles] osm_guwahati.pmtiles: {size_mb:.1f} MB')
    else:
        print('[PMTiles] Generation failed — will fall back to GeoJSON in Layer 5')
else:
    print('No OSM files available for PMTiles conversion')

## 5. Build tile manifest JSON for Layer 5

In [ ]:
# Build tile manifest JSON — the single config file Layer 5 reads
# Contains all tile URLs, visualization params, layer metadata

# Compute urban area stats for the manifest
urban_stats = {}
all_lulc = {**lulc_maps, **predicted}
for year in sorted(all_lulc.keys(), key=int):
    area = ee.Number(all_lulc[year].eq(0).multiply(ee.Image.pixelArea()).reduceRegion(
        ee.Reducer.sum(), aoi, SCALE, maxPixels=1e9).get('lulc')).divide(1e6).getInfo()
    urban_stats[year] = round(area, 2)
    print(f'Urban {year}: {area:.2f} km²')

manifest = {
    'project': 'Guwahati Urban Development Prediction Platform',
    'version': '1.0.0-poc',
    'center': CENTER,
    'zoom': 12,
    'bounds': [BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']],
    'epochs': {
        'historical': ['2000', '2005', '2010', '2016', '2021'],
        'predicted': ['2026', '2031', '2036'],
    },
    'urban_area_km2': urban_stats,
    'lulc': {
        'classes': LULC_LABELS,
        'palette': ['#' + c for c in LULC_PALETTE],
    },
    'tile_layers': {
        # LULC layers
        **{f'lulc_{y}': {
            'url': tile_urls[f'lulc_{y}'],
            'type': 'raster',
            'group': 'lulc',
            'label': f'LULC {y}' + (' (predicted)' if int(y) > 2021 else ''),
            'opacity': 0.7,
        } for y in sorted(all_lulc.keys(), key=int)},

        # Urban gain layers
        **{k: {
            'url': v,
            'type': 'raster',
            'group': 'change',
            'label': k.replace('urban_gain_', 'Urban Growth ').replace('_', '→'),
            'opacity': 0.8,
        } for k, v in tile_urls.items() if k.startswith('urban_gain')},

        # VIIRS
        **{f'viirs_{y}': {
            'url': tile_urls[f'viirs_{y}'],
            'type': 'raster',
            'group': 'nightlights',
            'label': f'Night Lights {y}',
            'opacity': 0.7,
        } for y in [2016, 2019, 2021, 2023]},

        # Analytical layers
        'suitability_2021': {
            'url': tile_urls['suitability_2021'],
            'type': 'raster',
            'group': 'analysis',
            'label': 'Urban Growth Suitability',
            'opacity': 0.7,
        },
        'ndvi_2021': {
            'url': tile_urls['ndvi_2021'],
            'type': 'raster',
            'group': 'analysis',
            'label': 'Vegetation Index (NDVI)',
            'opacity': 0.7,
        },
        'elevation': {
            'url': tile_urls['elevation'],
            'type': 'raster',
            'group': 'terrain',
            'label': 'Elevation (SRTM)',
            'opacity': 0.6,
        },
        'slope': {
            'url': tile_urls['slope'],
            'type': 'raster',
            'group': 'terrain',
            'label': 'Slope',
            'opacity': 0.6,
        },
    },
    'vector_layers': {
        'osm': {
            'type': 'pmtiles',
            'url': 'osm_guwahati.pmtiles',
            'layers': list(osm_files.keys()) if osm_files else [],
        }
    },
}

# Save manifest
with open('tile_manifest.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'\nManifest saved: tile_manifest.json')
print(f'  {len(manifest["tile_layers"])} raster tile layers')
print(f'  {len(manifest["vector_layers"]["osm"]["layers"])} vector layers in PMTiles')

# Also save to Drive for persistence
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
    import shutil
    dest = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
    os.makedirs(dest, exist_ok=True)
    shutil.copy('tile_manifest.json', f'{dest}/tile_manifest.json')
    if os.path.exists('osm_guwahati.pmtiles'):
        shutil.copy('osm_guwahati.pmtiles', f'{dest}/osm_guwahati.pmtiles')
    for f_name in osm_files.values():
        if os.path.exists(f_name):
            shutil.copy(f_name, f'{dest}/{f_name}')
    print(f'Files copied to Drive: {dest}/')
except Exception as e:
    print(f'Drive copy skipped: {e}')

## 6. Firebase Hosting setup

In [ ]:
# Firebase Hosting setup for static deployment
# Free tier: 10 GB storage, 360 MB/day transfer — plenty for a PoC

# Generate firebase.json config
firebase_config = {
    "hosting": {
        "public": "public",
        "ignore": ["firebase.json", "**/.*", "**/node_modules/**"],
        "headers": [
            {
                "source": "**/*.pmtiles",
                "headers": [
                    {"key": "Access-Control-Allow-Origin", "value": "*"},
                    {"key": "Content-Type", "value": "application/octet-stream"},
                    {"key": "Cache-Control", "value": "public, max-age=86400"}
                ]
            },
            {
                "source": "**/*.json",
                "headers": [
                    {"key": "Access-Control-Allow-Origin", "value": "*"},
                    {"key": "Cache-Control", "value": "public, max-age=3600"}
                ]
            },
            {
                "source": "**/*.geojson",
                "headers": [
                    {"key": "Access-Control-Allow-Origin", "value": "*"},
                    {"key": "Cache-Control", "value": "public, max-age=86400"}
                ]
            }
        ]
    }
}

with open('firebase.json', 'w') as f:
    json.dump(firebase_config, f, indent=2)
print('firebase.json generated')

# Create deployment directory structure
os.makedirs('public/data', exist_ok=True)

# Copy assets to public/
import shutil
shutil.copy('tile_manifest.json', 'public/tile_manifest.json')
if os.path.exists('osm_guwahati.pmtiles'):
    shutil.copy('osm_guwahati.pmtiles', 'public/data/osm_guwahati.pmtiles')
for f_name in osm_files.values():
    if os.path.exists(f_name):
        shutil.copy(f_name, f'public/data/{f_name}')

print('\nDeployment structure:')
for root, dirs, files in os.walk('public'):
    level = root.replace('public', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in files:
        size = os.path.getsize(os.path.join(root, file)) / 1024
        print(f'{subindent}{file} ({size:.0f} KB)')

print('''
To deploy:
  1. Install Firebase CLI: npm install -g firebase-tools
  2. Login: firebase login
  3. Init project: firebase init hosting (select your project)
  4. Copy public/ contents to the hosting directory
  5. Deploy: firebase deploy --only hosting

Or use GitHub Pages — just push public/ to a gh-pages branch.
''')

## 7. Verify tile endpoints

In [ ]:
# Verify tile endpoints are accessible

import urllib.request
import math

print('=== Tile Endpoint Verification ===\n')

# Compute correct tile coordinates for Guwahati center at zoom 12
lat, lon, zoom = 26.1445, 91.7362, 12
n = 2 ** zoom
tile_x = int((lon + 180.0) / 360.0 * n)
lat_rad = math.radians(lat)
tile_y = int((1.0 - math.asinh(math.tan(lat_rad)) / math.pi) / 2.0 * n)

# Test a few GEE tile URLs
test_layers = ['lulc_2021', 'lulc_2036', 'suitability_2021', 'viirs_2023']
for layer_name in test_layers:
    if layer_name not in tile_urls:
        print(f'  {layer_name}: NOT IN MANIFEST')
        continue
    url = tile_urls[layer_name].replace('{x}', str(tile_x)).replace('{y}', str(tile_y)).replace('{z}', str(zoom))
    try:
        req = urllib.request.Request(url, method='HEAD')
        resp = urllib.request.urlopen(req, timeout=10)
        status = resp.getcode()
        ctype = resp.headers.get('Content-Type', '?')
        print(f'  {layer_name}: HTTP {status} ({ctype})')
    except Exception as e:
        print(f'  {layer_name}: {e}')

# Check PMTiles file
if os.path.exists('osm_guwahati.pmtiles'):
    try:
        from pmtiles.reader import Reader, MmapSource
        with open('osm_guwahati.pmtiles', 'rb') as f:
            reader = Reader(MmapSource(f))
            header = reader.header()
            # pmtiles v3 uses min_longitude/min_latitude keys
            min_z = header.get('min_zoom', header.get('minzoom', '?'))
            max_z = header.get('max_zoom', header.get('maxzoom', '?'))
            min_lon = header.get('min_longitude', header.get('min_lon', '?'))
            min_lat = header.get('min_latitude', header.get('min_lat', '?'))
            max_lon = header.get('max_longitude', header.get('max_lon', '?'))
            max_lat = header.get('max_latitude', header.get('max_lat', '?'))
            print(f'\n  PMTiles: z{min_z}–z{max_z}, '
                  f'bounds=[{min_lon},{min_lat},{max_lon},{max_lat}]')
    except Exception as e:
        print(f'\n  PMTiles header read error: {e}')
else:
    print('\n  PMTiles: file not found (OSM vectors will use GeoJSON fallback)')

# Summary
print(f'''
=== Layer 4 Summary ===
  GEE raster tile layers: {len(tile_urls)}
  OSM vector PMTiles:     {'yes' if os.path.exists('osm_guwahati.pmtiles') else 'no (GeoJSON fallback)'}
  Manifest:               tile_manifest.json
  Deploy target:          Firebase Hosting / GitHub Pages
  
  GEE tile URLs are session-scoped (expire after ~24h).
  For production, export rasters to Cloud Storage + use rio-tiler.
  For this PoC, re-run this notebook to refresh tile URLs before using Layer 5.
''')